# NOORIX-AI PKR Currency Classifier — MobileNetV2
**Prepared by Muhammad Farhan**

Train a lightweight image classifier from a Kaggle dataset arranged as one folder per class. This is denomination/side classification only; it does not authenticate genuine notes or detect counterfeit notes.

## Kaggle inputs to add
1. Your already-used **Pakistani Currency Notes Image dataset** (Kaggle owner: `muhammadammarabid`; dataset slug: `pakistanicurrencydataset`).
2. Your second already-added Afghani & PKR banknotes dataset, if you want to include it. Do not combine Afghani and PKR classes in one NOORIX PKR classifier unless you deliberately want both currencies.

The notebook auto-searches image folders. It prints discovered class counts before training. Check labels carefully and exclude generic `Back` and `Front` folders because they do not identify a denomination.

## Kaggle notebook title
`NOORIX-PKR-Currency-MobileNetV2-Muhammad-Farhan`


In [ ]:
!pip -q install onnx onnxscript
import os, glob, json, shutil, random
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, accuracy_score
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
ROOTS=[Path(p) for p in glob.glob("/kaggle/input/*")]
for p in ROOTS: print(p)


In [ ]:
# Find candidate ImageFolder roots. Select one after reviewing the list.
IMG_EXTS={".jpg",".jpeg",".png",".bmp",".webp"}
candidates=[]
for root in ROOTS:
    for d in [root]+[Path(x) for x in glob.glob(str(root)+"/**",recursive=True) if Path(x).is_dir()]:
        sub=[x for x in d.iterdir() if x.is_dir()] if d.exists() else []
        if len(sub)>=2 and any(any(f.suffix.lower() in IMG_EXTS for f in x.rglob("*")) for x in sub):
            candidates.append(d)
candidates=list(dict.fromkeys(candidates))
for i,p in enumerate(candidates[:80]): print(i,p)
DATA_DIR="" # Set exact printed path, e.g. "/kaggle/input/pakistanicurrencydataset/data-rescaled"
if not DATA_DIR: raise ValueError("Set DATA_DIR to the folder whose immediate subfolders are class labels.")


In [ ]:
# Inspect folders and remove ambiguous generic labels by default.
from torchvision.datasets import ImageFolder
EXCLUDE_CLASSES={"Back","Front"}  # These do not identify a denomination.
IMG_SIZE=224
base_tfms=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
full=ImageFolder(DATA_DIR,transform=base_tfms)
all_names=full.classes
keep=[i for i,n in enumerate(all_names) if n not in EXCLUDE_CLASSES]
print("All classes:",all_names)
print("Included:",[all_names[i] for i in keep])
if len(keep)<2: raise ValueError("Need at least two meaningful denomination classes.")
# Filter samples while retaining ImageFolder transforms.
from torch.utils.data import Subset
indices=[i for i,(_,label) in enumerate(full.samples) if label in keep]
filtered=Subset(full,indices)
labels=[all_names[i] for i in keep]
old_to_new={old:new for new,old in enumerate(keep)}
targets=[old_to_new[full.samples[i][1]] for i in indices]
from sklearn.model_selection import train_test_split
tr_idx,va_idx=train_test_split(np.arange(len(indices)),test_size=.2,random_state=42,stratify=targets)
train_ds=Subset(filtered,tr_idx.tolist()); val_ds=Subset(filtered,va_idx.tolist())


In [ ]:
# Build class-balanced training sampler and MobileNetV2.
from collections import Counter
device="cuda" if torch.cuda.is_available() else "cpu"
train_targets=[targets[i] for i in tr_idx]
counts=Counter(train_targets)
weights=torch.DoubleTensor([1.0/counts[y] for y in train_targets])
sampler=torch.utils.data.WeightedRandomSampler(weights,len(weights),replacement=True)
train_loader=DataLoader(train_ds,batch_size=32,sampler=sampler,num_workers=2)
val_loader=DataLoader(val_ds,batch_size=32,shuffle=False,num_workers=2)
model=models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model.classifier[1]=nn.Linear(model.last_channel,len(labels))
model.to(device)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.AdamW(model.parameters(),lr=2e-4)
best=0.0


In [ ]:
# Train for up to 8 epochs; best validation accuracy checkpoint is retained.
for epoch in range(8):
    model.train(); running=0
    for x,y in train_loader:
        x=x.to(device); y=torch.tensor([targets[i] for i in []],device=device) if False else y.to(device)
        optimizer.zero_grad(); loss=criterion(model(x),y); loss.backward(); optimizer.step()
        running+=loss.item()
    model.eval(); yt=[]; yp=[]
    with torch.no_grad():
        for x,y in val_loader:
            out=model(x.to(device)); yp.extend(out.argmax(1).cpu().tolist()); yt.extend(y.tolist())
    acc=accuracy_score(yt,yp)
    print(f"Epoch {epoch+1}: train_loss={running/max(1,len(train_loader)):.4f} val_acc={acc:.4f}")
    if acc>best:
        best=acc; torch.save(model.state_dict(),"/kaggle/working/mobilenetv2_pkr_best.pth")
print("Best validation accuracy:",best)
print(classification_report(yt,yp,target_names=labels,zero_division=0))


In [ ]:
# Export and package. Keep ONNX and its external data file together if both are created.
model.load_state_dict(torch.load("/kaggle/working/mobilenetv2_pkr_best.pth",map_location=device,weights_only=True))
model.eval()
out=Path("/kaggle/working/NOORIX_PKR_MobileNetV2_Muhammad_Farhan"); out.mkdir(exist_ok=True)
torch.save(model.state_dict(),out/"mobilenetv2_pkr_best.pth")
(out/"labels.json").write_text(json.dumps({"classes":labels},ensure_ascii=False,indent=2),encoding="utf-8")
meta={"project":"NOORIX-AI","prepared_by":"Muhammad Farhan","architecture":"MobileNetV2","input_size":[1,3,224,224],
"normalization":{"mean":[0.485,0.456,0.406],"std":[0.229,0.224,0.225]},"validation_accuracy":float(best),
"warning":"Classification only; not banknote authentication."}
(out/"metadata.json").write_text(json.dumps(meta,indent=2),encoding="utf-8")
dummy=torch.randn(1,3,224,224,device=device)
torch.onnx.export(model,dummy,str(out/"mobilenetv2_pkr.onnx"),input_names=["image"],output_names=["logits"],
                  dynamic_axes={"image":{0:"batch"},"logits":{0:"batch"}},opset_version=18,dynamo=False)
zip_path=shutil.make_archive("/kaggle/working/NOORIX_PKR_MobileNetV2_Muhammad_Farhan","zip",out)
print("ZIP:",zip_path); print(*[p.name for p in out.iterdir()],sep="\n")
